# Chapter 18 - Model Selection

In this chapter, we move from simple model evaluation to reliable model selection.

A single train-test split is useful for a quick check, but it can depend too much on chance.

We will use cross-validation, hyperparameter tuning, and leakage-safe workflows to make model scores more trustworthy.

## Step 1 - Load the Dataset Once

We will use scikit-learn's Breast Cancer dataset. 

It is a binary classification dataset with 569 rows.

We will reuse the same dataset throughout the code so the different validation methods are easy to compare.

In [19]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load the complete dataset
data = load_breast_cancer()
X, y = data.data, data.target

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nClass names:", data.target_names)


Feature matrix shape: (569, 30)
Target vector shape: (569,)

Class names: ['malignant' 'benign']


The dataset has 30 numeric input features and one binary target. 

Because this is a classification problem, we use stratified splitting so both classes stay represented in train and test data.

In [20]:
import numpy as np

# Keep one untouched test set for the final check after each validation method.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("\nTraining class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Testing class counts:", dict(zip(*np.unique(y_test, return_counts=True))))


Training rows: 455
Testing rows: 114

Training class counts: {np.int64(0): np.int64(170), np.int64(1): np.int64(285)}
Testing class counts: {np.int64(0): np.int64(42), np.int64(1): np.int64(72)}


## Step 2 - Leave-One-Out Cross Validation

Leave-One-Out Cross Validation trains one model for each training example. 

Each time, one row becomes the validation row and all remaining rows become the training data.

This uses the data fully, but it is expensive because it trains many models.

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import LeaveOneOut, cross_val_score

loo = LeaveOneOut()
loocv_model = LogisticRegression(max_iter=10000)

# LOOCV runs one validation round for every row in the training set.
loocv_scores = cross_val_score(
    loocv_model,
    X_train,
    y_train,
    cv=loo,
    scoring="accuracy",
    n_jobs=-1,
)

print("Number of LOOCV iterations:", len(loocv_scores))
print("Mean LOOCV accuracy:", loocv_scores.mean())


Number of LOOCV iterations: 455
Mean LOOCV accuracy: 0.9516483516483516


The number of iterations equals the number of training rows. 

The mean score is the average of many tiny validation tests, so it is less dependent on one random split.

In [22]:
# After validation, train one final model on the full training set.
loocv_model.fit(X_train, y_train)
loocv_pred = loocv_model.predict(X_test)

print("Final test accuracy after LOOCV estimate:", accuracy_score(y_test, loocv_pred))


Final test accuracy after LOOCV estimate: 0.9649122807017544


The final test set was not used during LOOCV. This keeps the final accuracy as an unbiased check on unseen data.

## Step 3 - K-Fold Cross Validation

K-Fold Cross Validation is a more practical version of the same idea. 

Instead of training one model per row, we split the training data into K folds and train K models.

Here we use 5 folds, so each fold gets one turn as validation data.

In [23]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_model = LogisticRegression(max_iter=10000)

kfold_scores = cross_val_score(
    kfold_model,
    X_train,
    y_train,
    cv=kf,
    scoring="accuracy",
    n_jobs=-1, 
)

print("\nFold-wise accuracy:", kfold_scores)
print("Mean K-Fold accuracy:", kfold_scores.mean())



Fold-wise accuracy: [0.92307692 0.97802198 0.97802198 0.94505495 0.93406593]
Mean K-Fold accuracy: 0.9516483516483516


The fold-wise scores show how the same model behaves on different validation folds. 

The mean score is a more stable estimate than one train-test split.

In [24]:
kfold_model.fit(X_train, y_train)
kfold_pred = kfold_model.predict(X_test)

print("Final test accuracy after K-Fold estimate:", accuracy_score(y_test, kfold_pred))


Final test accuracy after K-Fold estimate: 0.9649122807017544


If the final test accuracy is close to the cross-validation mean, the validation estimate was consistent with the unseen test set.

## Step 4 - Stratified K-Fold Cross Validation

Regular K-Fold can create folds with uneven class balance. 

Stratified K-Fold fixes this by preserving the class ratio inside every fold.

This is usually the better default for classification problems.

In [25]:
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_model = LogisticRegression(max_iter=10000)

stratified_scores = cross_val_score(
    stratified_model,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,
)

print("Fold-wise accuracy:", stratified_scores)
print("Mean Stratified K-Fold accuracy:", stratified_scores.mean())


Fold-wise accuracy: [0.96703297 0.92307692 0.94505495 0.95604396 0.93406593]
Mean Stratified K-Fold accuracy: 0.945054945054945


Stratification keeps every validation fold representative of the whole classification problem. 

This makes metrics such as accuracy, precision, recall, and F1 more trustworthy.

In [26]:
stratified_model.fit(X_train, y_train)
stratified_pred = stratified_model.predict(X_test)

print("Final test accuracy after Stratified K-Fold estimate:", accuracy_score(y_test, stratified_pred))
print("\nClassification report:\n", classification_report(y_test, stratified_pred, target_names=data.target_names))


Final test accuracy after Stratified K-Fold estimate: 0.9649122807017544

Classification report:
               precision    recall  f1-score   support

   malignant       0.97      0.93      0.95        42
      benign       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



The classification report gives more detail than accuracy alone. 

It shows whether the model is performing well for both classes, not just overall.

## Step 5 - Manual Hyperparameter Tuning

Hyperparameters are settings we choose before training. 

In KNN, the number of neighbors is a hyperparameter.

We can tune it manually by trying a few values and comparing the test accuracy.

In [27]:
from sklearn.neighbors import KNeighborsClassifier

for k in [3, 5, 7, 9]:
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train, y_train)
    knn_pred = knn_model.predict(X_test)
    print(f"k = {k}, Accuracy = {accuracy_score(y_test, knn_pred):.3f}")


k = 3, Accuracy = 0.930
k = 5, Accuracy = 0.912
k = 7, Accuracy = 0.930
k = 9, Accuracy = 0.939


This simple loop shows that model performance can change when we change a hyperparameter. 

Manual tuning is useful for intuition, but it becomes slow when there are many settings.

## Step 6 - Grid Search with Cross-Validation

Grid Search tries every combination in a parameter grid. 

Here we tune a Decision Tree using 4 choices for max depth, 2 choices for min samples split, and 2 choices for criterion.

That gives 16 combinations, and each combination is evaluated with 5-fold cross-validation.

In [28]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Start with a Decision Tree model and define the exact grid to search.
tree_model = DecisionTreeClassifier(random_state=42)
param_grid = {
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5],
    "criterion": ["gini", "entropy"],
}

print("Number of grid combinations:", 4 * 2 * 2)


Number of grid combinations: 16


In [29]:
grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)


Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 5}
Best cross-validation score: 0.9384615384615385


GridSearchCV chooses the parameter combination with the best average validation score across the folds. 

The selected estimator is already refit on the full training data.

In [30]:
grid_pred = grid_search.predict(X_test)
print("Final test accuracy after Grid Search:", accuracy_score(y_test, grid_pred))


Final test accuracy after Grid Search: 0.9210526315789473


The final test accuracy tells us how the tuned Decision Tree performs on data that was not used during the grid search.

## Step 7 - Random Search with Cross-Validation

Random Search is useful when the search space is large. 

Instead of checking every possible combination, it samples a fixed number of random combinations.

This gives us a time budget through n_iter while still exploring a broad range of settings.

In [31]:
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

forest_model = RandomForestClassifier(random_state=42)

# Random Search can sample integer ranges instead of requiring fixed lists for every setting.
param_dist = {
    "n_estimators": randint(50, 500),
    "max_depth": randint(5, 50),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
}

random_search = RandomizedSearchCV(
    estimator=forest_model,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    verbose=1,
    n_jobs=-1,
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best cross-validation score:", random_search.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'max_depth': 9, 'max_features': 'log2', 'min_samples_leaf': 7, 'min_samples_split': 10, 'n_estimators': 77}
Best cross-validation score: 0.9560439560439562


Random Search does not promise that it checked the absolute best combination. 

Its advantage is that it can find strong settings much faster than an exhaustive grid search.

In [32]:
random_pred = random_search.predict(X_test)
print("Final test accuracy after Random Search:", accuracy_score(y_test, random_pred))


Final test accuracy after Random Search: 0.9298245614035088


## Step 8 - Data Leakage and Safe Workflows

Data leakage happens when the model gets information that would not be available at prediction time.

- A common mistake is fitting preprocessing steps before the train-test split or outside the cross-validation loop.

### Wrong Pattern - Fit Preprocessing Before Splitting

The following pattern is risky because the scaler learns from the full dataset before the test data is separated.

Since this is wrong way , so we just keep it as comments.

In [33]:
# from sklearn.preprocessing import StandardScaler
#
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)  # Wrong: this uses all rows, including future test rows.
# X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)


### Correct Pattern - Split First, Then Fit on Training Data

We first split the raw data. 

Then the scaler learns only from the training rows and transforms the test rows using those training statistics.

In [34]:
from sklearn.preprocessing import StandardScaler

leak_safe_X_train, leak_safe_X_test, leak_safe_y_train, leak_safe_y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Before scaling, first training row values:")
print(leak_safe_X_train[0, :5])

scaler = StandardScaler()
leak_safe_X_train_scaled = scaler.fit_transform(leak_safe_X_train)
leak_safe_X_test_scaled = scaler.transform(leak_safe_X_test)

print("\nAfter scaling, first training row values:")
print(leak_safe_X_train_scaled[0, :5])


Before scaling, first training row values:
[1.032e+01 1.635e+01 6.531e+01 3.249e+02 9.434e-02]

After scaling, first training row values:
[-1.07200079 -0.6584246  -1.0880801  -0.93927364 -0.13593988]


The important detail is not the changed numbers. 

The important detail is that fit_transform was used only on the training data, while the test data used transform.

### Safest Pattern - Put Preprocessing Inside a Pipeline

In cross-validation, preprocessing should happen separately inside every fold. 

A Pipeline makes that automatic because each fold fits the scaler only on that fold's training portion.

In [35]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

pipeline_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
)

print("Pipeline fold-wise accuracy:", pipeline_scores)
print("Pipeline mean accuracy:", pipeline_scores.mean())


Pipeline fold-wise accuracy: [0.96703297 0.98901099 0.97802198 0.98901099 0.96703297]
Pipeline mean accuracy: 0.9780219780219781


The Pipeline keeps preprocessing and model training together. 

This prevents validation rows from influencing scaling statistics during cross-validation.

## Chapter Recap

In this chapter we learnt about various ways of doing model selection:

- Cross-validation gives a more reliable performance estimate than one random split.
- LOOCV uses every sample as validation once, but it can be expensive.
- K-Fold is the practical default for many problems.
- Stratified K-Fold is preferred for classification because it preserves class balance.
- Hyperparameter tuning searches for better model settings.
- Grid Search is exhaustive, while Random Search is faster for large spaces.
- Data leakage can make scores look better than reality, so splitting early and using Pipelines is essential.